# บทที่ 5 · Error, ลองใหม่, log และการเก็บรหัสผ่าน

โปรแกรมที่ทำงานกับข้อมูลจริงต้องเจอปัญหาแน่นอน: ไฟล์ยังไม่มา, เน็ตหลุด, ข้อมูลผิดรูปแบบ
สิ่งสำคัญคือ **รู้ตัวเมื่อมีปัญหา** และ **จัดการให้ถูกวิธี**

บทนี้: **อ่าน error · try/except · raise · ลองใหม่ (retry) · logging · ค่าตั้งค่าและรหัสผ่านใน .env**

## 5.1 อ่าน error ให้เป็น

**อ่านบรรทัดสุดท้ายก่อนเสมอ** : `ชนิดของ error: ข้อความ` แล้วค่อยไล่ขึ้นไปหาบรรทัดที่เป็นโค้ดของเรา

In [ ]:
def to_number(text):
    return int(text)


to_number("สองร้อย")        # ❌ ValueError: invalid literal for int() with base 10: 'สองร้อย'

| error | ตัวอย่างสาเหตุ |
|---|---|
| `ValueError` | แปลงไม่ได้ `int("สองร้อย")` |
| `KeyError` | dict ไม่มี key นั้น `friend["email"]` |
| `TypeError` | ใช้ผิดชนิด `"ราคา " + 55` |
| `ZeroDivisionError` | หารด้วย 0 |
| `FileNotFoundError` | ไม่มีไฟล์ / path ผิด |
| `ModuleNotFoundError` | ยังไม่ได้ติดตั้ง library / เลือก kernel ผิด |

## 5.2 `try / except` : ลองทำ ถ้าพังให้ทำอีกทาง

In [ ]:
def safe_number(text):
    try:
        return int(text)
    except ValueError:
        print(f"  '{text}' ไม่ใช่ตัวเลข")
        return None


for value in ["120", "สองร้อย", "0", "-5"]:
    print(value, "->", safe_number(value))

In [ ]:
# finally = ทำเสมอไม่ว่าจะสำเร็จหรือพัง (เช่น ปิดร้าน ล็อกประตู)
def open_shop(has_key):
    try:
        if not has_key:
            raise PermissionError("ลืมกุญแจ")
        print("เปิดร้านแล้ว")
    except PermissionError as problem:
        print("เปิดไม่ได้:", problem)
    finally:
        print("ส่งข้อความบอกลูกน้องแล้ว")


open_shop(True)
open_shop(False)

## 5.3 `raise` : สั่งให้หยุดพร้อมบอกเหตุผล

เมื่อเจอข้อมูลที่ไม่ควรเป็นแบบนั้น **อย่าปล่อยผ่านเงียบ ๆ** ให้ `raise` พร้อมข้อความที่บอกชัดว่าผิดอะไร

In [ ]:
def withdraw(balance: float, amount: float) -> float:
    if amount <= 0:
        raise ValueError(f"จำนวนเงินต้องมากกว่า 0 (ได้ {amount})")
    if amount > balance:
        raise ValueError(f"เงินไม่พอ: มี {balance} จะถอน {amount}")
    return balance - amount


print(withdraw(1000, 300))

In [ ]:
withdraw(1000, 5000)          # ❌ ตั้งใจให้ error

In [ ]:
# ตรวจว่าข้อมูลมีครบไหม: set ลบกัน = สิ่งที่ขาด
REQUIRED = {"name", "phone", "address"}


def check_delivery(order: dict) -> None:
    missing = REQUIRED - set(order)
    if missing:
        raise ValueError(f"ข้อมูลส่งของไม่ครบ ขาด: {sorted(missing)}")
    print("พร้อมส่ง", order["name"])


check_delivery({"name": "มานี", "phone": "081", "address": "ขอนแก่น"})
try:
    check_delivery({"name": "ชูใจ"})
except ValueError as problem:
    print("ส่งไม่ได้:", problem)

⚠️ **อย่าเขียนแบบนี้**: error หายเงียบ ไม่มีใครรู้ว่าข้อมูลหาย

```python
try:
    save_orders()
except Exception:
    pass
```

ถ้าจับ error แล้ว ให้ทำอะไรสักอย่าง (log / ย้ายไฟล์เสียไปแยกไว้) และถ้างานนั้น **ต้องถือว่าล้มเหลว** ให้ `raise` ต่อ

## 5.4 ลองใหม่ (retry)

บางปัญหาเป็นแค่ **ชั่วคราว**: เน็ตกระตุก, เว็บคนใช้เยอะ → รอสักครู่แล้วลองใหม่มักสำเร็จ
แต่บางปัญหา **ลองกี่ครั้งก็ไม่หาย**: รหัสผ่านผิด, ข้อมูลผิดรูปแบบ → ควรหยุดแล้วแจ้งทันที

In [ ]:
import random
import time

random.seed(3)


def send_line_message():
    if random.random() < 0.6:
        raise ConnectionError("เน็ตหลุด")
    return "ส่งสำเร็จ"


def send_with_retry(max_tries: int = 5):
    for attempt in range(1, max_tries + 1):
        try:
            return send_line_message()
        except ConnectionError as problem:
            if attempt == max_tries:
                raise                               # ลองครบแล้วยังพัง -> ส่ง error ต่อ
            wait = 0.2 * 2 ** attempt               # รอนานขึ้นเรื่อย ๆ 0.4, 0.8, 1.6 ...
            print(f"ครั้งที่ {attempt}: {problem} รอ {wait:.1f} วินาทีแล้วลองใหม่")
            time.sleep(wait)


print(send_with_retry())

## 5.5 logging : จดบันทึกแทน print

`print` แค่แสดงข้อความ ส่วน `logging` มี **เวลา** และ **ระดับความสำคัญ** กำกับ ย้อนดูภายหลังได้ว่าเกิดอะไรขึ้นตอนไหน

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
log = logging.getLogger("coffee_shop")

log.info("เปิดร้าน ขายได้ %d แก้ว", 12)
log.warning("นมเหลือ %d กล่อง", 2)
log.error("เครื่องชงกาแฟเสีย")
log.debug("ข้อความละเอียด (ไม่แสดง เพราะตั้ง level=INFO)")

ระดับ: `DEBUG` < `INFO` < `WARNING` < `ERROR` ตั้ง level ไว้ระดับไหน จะแสดงตั้งแต่ระดับนั้นขึ้นไป

## 5.6 ค่าตั้งค่าและรหัสผ่าน: ห้ามเขียนลงในโค้ด

โค้ดจะถูกแชร์/ขึ้น git รหัสผ่านที่เขียนในโค้ดจะหลุดไปด้วย
วิธีมาตรฐาน: เก็บไว้ใน **environment variable** (ตัวแปรของระบบ) หรือไฟล์ **`.env`** ที่ไม่ขึ้น git แล้วให้โค้ดอ่านมาใช้

ไฟล์ `.env` ของ repo นี้ (คัดลอกมาจาก `.env.example` ตอน setup):

```
RUSTFS_ACCESS_KEY=student
RUSTFS_SECRET_KEY=classroom
SHOP_API_KEY=class-demo-key
```

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()                                    # อ่าน .env (หาไฟล์จากโฟลเดอร์ปัจจุบันขึ้นไปเรื่อย ๆ)

api_url = os.getenv("SHOP_API_URL", "http://127.0.0.1:8000")      # (ชื่อ, ค่าสำรองถ้าไม่มี)
api_key = os.getenv("SHOP_API_KEY")
print("API:", api_url)
print("มี key ไหม:", api_key is not None)       # อย่า print รหัสผ่านออกมาจริง ๆ
print(os.getenv("NOT_EXIST"))                   # ไม่มี -> None

## ✅ สรุปบทที่ 5

| เรื่อง | ตัวอย่าง |
|---|---|
| อ่าน error | ดูบรรทัดสุดท้าย `ValueError: ...` |
| จับ error | `try: ... except ValueError as problem: ...  finally: ...` |
| แจ้ง error เอง | `raise ValueError("ข้อความที่บอกว่าผิดอะไร")` |
| ลองใหม่ | loop + `time.sleep()` เฉพาะปัญหาชั่วคราว |
| log | `log = logging.getLogger("ชื่อ")`, `log.info(...)`, `log.warning(...)` |
| รหัสผ่าน | เก็บใน `.env` → `load_dotenv()` → `os.getenv("ชื่อ")` |

🧪 **แบบฝึกท้ายบท:** เขียน `clean_scores(scores: list[str]) -> list[int]`
แปลงคะแนนสอบ (ข้อความ) เป็นตัวเลข เก็บเฉพาะที่อยู่ระหว่าง 0–100 ตัวที่แปลงไม่ได้หรืออยู่นอกช่วงให้ข้ามและ `log.warning`
และถ้า `scores` ว่างให้ `raise ValueError`

In [ ]:
scores = ["85", "92", "หนึ่งร้อย", "105", "70", "-3"]
# เขียนต่อตรงนี้